# Module 08: LLD Financial Engines Splitwise Rate Limiting — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/splitwise_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import splitwise_engine

classes = [n for n, o in inspect.getmembers(splitwise_engine, inspect.isclass)
           if o.__module__ == 'splitwise_engine']
functions = [n for n, o in inspect.getmembers(splitwise_engine, inspect.isfunction)
             if o.__module__ == 'splitwise_engine']

print('module   : splitwise_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(splitwise_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Splitwise equal split and penny rounding

This is the module's own `test_splitwise_equal_split_and_penny_rounding` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import time
from decimal import Decimal

from splitwise_engine import (
    Split,
    SplitType,
    SplitwiseLedger,
)

ledger = SplitwiseLedger()
# $100 split 3 ways: 33.33 + 33.33 + 33.34 = 100.00
ledger.add_expense(
    paid_by="Alice",
    total_amount=Decimal("100.00"),
    split_type=SplitType.EQUAL,
    participants=["Alice", "Bob", "Charlie"],
)

# Net sum of all balances in the system MUST strictly equal zero!
net_sum = sum(ledger.net_balances.values())
assert net_sum == Decimal("0.00")

# Alice paid 100, her share is 33.33 -> net is +66.67
assert ledger.net_balances["Alice"] == Decimal("66.67")
assert ledger.net_balances["Bob"] == Decimal("-33.33")
assert ledger.net_balances["Charlie"] == Decimal("-33.34")

print('PASSED: test_splitwise_equal_split_and_penny_rounding')

## 3. 🔮 Prediction — commit before you run

Three people split an expense unevenly. Predict whether the sum of all pairwise balances is exactly zero, and what floating-point issue could break that.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_splitwise_percentage_split`, which tests exactly this property.


In [ ]:
ledger = SplitwiseLedger()
ledger.add_expense(
    paid_by="David",
    total_amount=Decimal("200.00"),
    split_type=SplitType.PERCENTAGE,
    splits=[
        Split("David", Decimal("50.00")),
        Split("Emma", Decimal("30.00")),
        Split("Frank", Decimal("20.00")),
    ],
)

assert sum(ledger.net_balances.values()) == Decimal("0.00")
assert ledger.net_balances["David"] == Decimal("100.00")
assert ledger.net_balances["Emma"] == Decimal("-60.00")
assert ledger.net_balances["Frank"] == Decimal("-40.00")

print('PASSED: test_splitwise_percentage_split')

## 4. Measure it: Debt simplification minimizes transactions

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_debt_simplification_minimizes_transactions` and times it.


In [ ]:

_t0 = time.perf_counter()

ledger = SplitwiseLedger()
# B pays $10 for A
ledger.add_expense(
    paid_by="Bob",
    total_amount=Decimal("10.00"),
    split_type=SplitType.EXACT,
    splits=[Split("Alice", Decimal("10.00"))],
)
# C pays $10 for B
ledger.add_expense(
    paid_by="Charlie",
    total_amount=Decimal("10.00"),
    split_type=SplitType.EXACT,
    splits=[Split("Bob", Decimal("10.00"))],
)

# Bob's net balance should be 0.00 (+10 from A, -10 to C)
assert ledger.net_balances["Bob"] == Decimal("0.00")

transactions = ledger.simplify_debts()
assert len(transactions) == 1
assert transactions[0].debtor == "Alice"
assert transactions[0].creditor == "Charlie"
assert transactions[0].amount == Decimal("10.00")

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_debt_simplification_minimizes_transactions')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(splitwise_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Money is not a float. Use integer minor units or Decimal, always.
2. Balances must sum to zero - assert it, because a rounding drift is silent.
3. Simplifying debts is a graph problem, not an accounting one.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
